In [1]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

In [2]:
#!pip install "hopsworks[python]"

In [3]:
#!pip install "numpy<2.0.0"
#!pip install python-dotenv

In [4]:
import hopsworks

In [2]:
!ls work/AIEngineering

computervision	  mlops    requisitos		 statistic
machine_learning  my_work  start_jupyter_docker  univesp


In [5]:
import os
from dotenv import dotenv_values
ENV = dotenv_values("./work/AIEngineering/machine_learning/feature_store/hopsworks/.env")

In [6]:
project = hopsworks.login(
    project='my_first_project10',  
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=ENV['api_key']
)

2026-09-03 11:37:43,187 INFO: Initializing external client
2026-09-03 11:37:43,188 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443


2026-09-03 11:37:46,596 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42213


In [7]:
# Access the feature store
fs = project.get_feature_store()

In [6]:
fs.__dict__

{'_id': 30898,
 '_name': 'my_first_project10_featurestore',
 '_created': '2026-08-21T11:27:29.000Z',
 '_project_name': 'my_first_project10',
 '_project_id': 42213,
 '_online_feature_store_name': 'my_first_project10',
 '_online_feature_store_size': None,
 '_offline_feature_store_name': 'my_first_project10_featurestore',
 '_online_enabled': True,
 '_num_feature_groups': 0,
 '_num_training_datasets': 0,
 '_num_storage_connectors': 4,
 '_num_feature_views': 0,
 '_feature_group_api': <hsfs.core.feature_group_api.FeatureGroupApi at 0x7fa3f0ba0310>,
 '_storage_connector_api': <hsfs.core.storage_connector_api.StorageConnectorApi at 0x7fa3f3485690>,
 '_training_dataset_api': <hsfs.core.training_dataset_api.TrainingDatasetApi at 0x7fa3f0b82cd0>,
 '_feature_group_engine': <hsfs.core.feature_group_engine.FeatureGroupEngine at 0x7fa3f0114d10>,
 '_transformation_function_engine': <hsfs.core.transformation_function_engine.TransformationFunctionEngine at 0x7fa3f0841f10>,
 '_feature_view_engine': <hsfs

In [ ]:
import pandas as pd
data = {
    "pessoa_id": [1001, 1002, 1003, 1004], 
    "idade": [28, 45, 32, 19],
    "gastos_30d": [1250.50, 4300.00, 890.20, 150.00],
    "media_saldo_6m": [3200.00, 12500.00, 2100.00, 400.00],
    "dias_desde_ultimo_login": [1, 5, 2, 0]
}

df_pessoa = pd.DataFrame(data)

In [ ]:
import pandas as pd
data = {
    "pessoa_id": [1001, 1002, 1003, 1004],
    "peso": [110, 68, 90, 62],
    "time_futebol": ["palmeiras", "flamengo", "cruzeiro", "vasco"],
    "id_pais": ["Brasil", "Japao", "Espanha", "Brasil"],
    "quantos_dias_sem_comer": [1, 2, 4, 3]
}

df_identidade = pd.DataFrame(data)

df_financeiro = pd.DataFrame({
    "pessoa_id": [1001, 1002, 1003],
    "gastos_100d": [1200.0, 8500.0, 400.0],
    "inadimplente": [0, 0, 1] # Nosso Alvo (Target)
})

In [ ]:
df_pessoa

,pessoa_id,gastos_100d,inadimplente
0,1001,1200.0,0
1,1002,8500.0,0
2,1003,400.0,1


In [23]:
df_financeiro

,pessoa_id,gastos_100d,inadimplente
0,1001,1200.0,0
1,1002,8500.0,0
2,1003,400.0,1


In [10]:
pessoa_fg = fs.get_or_create_feature_group(
    name="pessoa_comportamento_financeiro",
    version=1,
    primary_key=["pessoa_id"],
    description="Comportamento financeiro e de engajamento da entidade Pessoa",
    online_enabled=True # Habilita acesso de baixa latência para predições em tempo real
)

pessoa_fg.insert(df_pessoa)

Uploading Dataframe: 100.00% |██████████| Rows 4/4 | Elapsed Time: 00:01 | Remaining Time: 00:00


(None, None)

In [11]:
fg_identidade = fs.get_or_create_feature_group(
    name="pessoa_comportamento_identitario",
    version=1,
    primary_key=["pessoa_id"],
    online_enabled=True
)
fg_identidade.insert(df_identidade)

Uploading Dataframe: 100.00% |██████████| Rows 4/4 | Elapsed Time: 00:01 | Remaining Time: 00:00


(None, None)

In [12]:
fg_financeiro = fs.get_or_create_feature_group(
    name="pessoa_comportamento_financeiro_plus",
    version=1,
    primary_key=["pessoa_id"],
    online_enabled=True
)
fg_financeiro.insert(df_financeiro)

Uploading Dataframe: 100.00% |██████████| Rows 3/3 | Elapsed Time: 00:01 | Remaining Time: 00:00


(None, None)

In [13]:
# Example: Read an existing feature group
fg_comportamento_financeiro = fs.get_feature_group(
    name="pessoa_comportamento_financeiro",
    version=1
)


fg_comportamento_identitario = fs.get_feature_group(
    name="pessoa_comportamento_identitario",
    version=1
)

fg_financeiro = fs.get_feature_group(
    name="pessoa_comportamento_financeiro_plus",
    version=1
)

In [ ]:
#propriedades da feature_group, existem outras (ver doc)
fg.columns
fg.description

[Feature('pessoa_id', 'bigint', None, True, False, False, 'bigint', None, 52432),
 Feature('idade', 'bigint', None, False, False, False, 'bigint', None, 52432),
 Feature('gastos_30d', 'double', None, False, False, False, 'double', None, 52432),
 Feature('media_saldo_6m', 'double', None, False, False, False, 'double', None, 52432),
 Feature('dias_desde_ultimo_login', 'bigint', None, False, False, False, 'bigint', None, 52432)]

'Comportamento financeiro e de engajamento da entidade Pessoa'

In [ ]:
# Read data from the feature group
df = fg.read()
print(f"Connected to {project.name}. Feature group has {len(df)} rows")

# For model serving
ms = project.get_model_serving()

# For model registry
mr = project.get_model_registry()

In [19]:
type(fg)

hsfs.feature_group.FeatureGroup

In [ ]:
fg

##  Feature View

In [39]:
all_fgs = fs.get_feature_groups()

In [47]:
for fg in all_fgs:
  print(f"Nome: {fg.name}, Versão: {fg.version}")

Nome: pessoa_comportamento_financeiro, Versão: 1
Nome: pessoa_comportamento_identitario, Versão: 1
Nome: pessoa_comportamento_financeiro_plus, Versão: 1


In [42]:
fg_comportamento_identitario.columns

[Feature('pessoa_id', 'bigint', None, True, False, False, 'bigint', None, 53307),
 Feature('peso', 'bigint', None, False, False, False, 'bigint', None, 53307),
 Feature('time_futebol', 'string', None, False, False, False, 'varchar(100)', None, 53307),
 Feature('id_pais', 'string', None, False, False, False, 'varchar(100)', None, 53307),
 Feature('quantos_dias_sem_comer', 'bigint', None, False, False, False, 'bigint', None, 53307)]

In [44]:
fg_comportamento_financeiro.columns

[Feature('pessoa_id', 'bigint', None, True, False, False, 'bigint', None, 52432),
 Feature('idade', 'bigint', None, False, False, False, 'bigint', None, 52432),
 Feature('gastos_30d', 'double', None, False, False, False, 'double', None, 52432),
 Feature('media_saldo_6m', 'double', None, False, False, False, 'double', None, 52432),
 Feature('dias_desde_ultimo_login', 'bigint', None, False, False, False, 'bigint', None, 52432)]

In [48]:
fg_financeiro.columns

[Feature('pessoa_id', 'bigint', None, True, False, False, 'bigint', None, 52433),
 Feature('gastos_100d', 'double', None, False, False, False, 'double', None, 52433),
 Feature('inadimplente', 'bigint', None, False, False, False, 'bigint', None, 52433)]

In [60]:
query_features = fg_comportamento_financeiro.select(["pessoa_id", "idade", "dias_desde_ultimo_login"]) \
    .join(fg_financeiro.select(["gastos_100d", "inadimplente"])) \
        .join(fg_comportamento_identitario.select(["peso", "quantos_dias_sem_comer"]))

In [61]:
query_features.__dict__

{'_feature_store_name': 'my_first_project10_featurestore',
 '_feature_store_id': 30898,
 '_left_feature_group': <hsfs.feature_group.FeatureGroup at 0x7fa3f0128d10>,
 '_left_features': [Feature('idade', None, None, False, False, False, None, None, None),
  Feature('dias_desde_ultimo_login', None, None, False, False, False, None, None, None)],
 '_left_feature_group_start_time': None,
 '_left_feature_group_end_time': None,
 '_joins': [<hsfs.constructor.join.Join at 0x7fa386ba0cd0>,
 '_filter': None,
 '_limit': None,
 '_lookback': None,
 '_python_engine': True,
 '_query_constructor_api': <hsfs.core.query_constructor_api.QueryConstructorApi at 0x7fa3f0116450>,
 '_storage_connector_api': <hsfs.core.storage_connector_api.StorageConnectorApi at 0x7fa386ba3910>,
 '_ambiguous_features_in_query': {}}

In [ ]:
query_features._left_features
query_features._left_feature_group.__dict__

[Feature('idade', None, None, False, False, False, None, None, None),
 Feature('dias_desde_ultimo_login', None, None, False, False, False, None, None, None)]

{'_version': 1,
 '_name': 'pessoa_comportamento_financeiro',
 '_event_time': None,
 '_online_enabled': True,
 '_location': 'hopsfs://rpc.namenode.service.consul:8020/apps/hive/warehouse/my_first_project10_featurestore.db/pessoa_comportamento_financeiro_1',
 '_id': 52432,
 '_subject': None,
 '_online_topic_name': 'my_first_project10_onlinefs',
 '_topic_name': None,
 '_notification_topic_name': None,
 '_deprecated': False,
 '_feature_store_id': 30898,
 '_feature_store': <hsfs.feature_store.FeatureStore at 0x7fa3f1091ed0>,
 '_variable_api': <hopsworks_common.core.variable_api.VariableApi at 0x7fa386bdf3d0>,
 '_alert_api': <hopsworks_common.core.alerts_api.AlertsApi at 0x7fa386bde850>,
 '_ttl': None,
 '_ttl_enabled': False,
 '_sink_enabled': False,
 '_missing_mandatory_tags': [],
 '_online_config': <hsfs.online_config.OnlineConfig at 0x7fa386bdf250>,
 '_data_source': <hsfs.core.data_source.DataSource at 0x7fa386bdf450>,
 '_multi_part_insert': False,
 '_embedding_index': None,
 '_expectatio

In [18]:
fs = project.get_feature_store()

In [61]:
feature_view = fs.create_feature_view(
    name="modelo_forecasting",
    version=2,
    labels=["inadimplente"],
    query=query_features
)

Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/42213/fs/30898/fv/modelo_forecasting/version/2


In [51]:
feature_view

In [62]:
feature_view = fs.get_feature_view(name="modelo_forecasting", version=2)

In [26]:
feature_view.__dict__
feature_view._query.__dict__

{'_name': 'modelo_forecasting',
 '_id': 30831,
 '_tags': [],
 '_query': <hsfs.constructor.query.Query at 0x7fee801c3f10>,
 '_featurestore_id': 30898,
 '_feature_store_id': 30898,
 '_feature_store_name': 'my_first_project10',
 '_version': 1,
 '_description': '',
 '_labels': ['inadimplente'],
 '_inference_helper_columns': [],
 '_training_helper_columns': [],
 '_transformation_functions': [],
 '_features': [Training Dataset Feature('idade', 'bigint', 0, False, idade, 52432, None),
  Training Dataset Feature('dias_desde_ultimo_login', 'bigint', 1, False, dias_desde_ultimo_login, 52432, None),
  Training Dataset Feature('gastos_100d', 'double', 2, False, gastos_100d, 52433, None),
  Training Dataset Feature('inadimplente', 'bigint', 3, True, inadimplente, 52433, None),
  Training Dataset Feature('peso', 'bigint', 4, False, peso, 53307, None),
  Training Dataset Feature('quantos_dias_sem_comer', 'bigint', 5, False, quantos_dias_sem_comer, 53307, None)],
 '_request_parameters': None,
 '_featu

{'_feature_store_name': 'my_first_project10_featurestore',
 '_feature_store_id': 30898,
 '_left_feature_group': <hsfs.feature_group.FeatureGroup at 0x7fee801d8dd0>,
 '_left_features': [Feature('idade', 'bigint', None, False, False, False, None, None, 52432),
  Feature('dias_desde_ultimo_login', 'bigint', None, False, False, False, None, None, 52432)],
 '_left_feature_group_start_time': None,
 '_left_feature_group_end_time': None,
 '_joins': [<hsfs.constructor.join.Join at 0x7fee801db9d0>,
 '_filter': None,
 '_limit': None,
 '_lookback': None,
 '_python_engine': True,
 '_query_constructor_api': <hsfs.core.query_constructor_api.QueryConstructorApi at 0x7fee801c3f90>,
 '_storage_connector_api': <hsfs.core.storage_connector_api.StorageConnectorApi at 0x7fee801c3f50>,
 '_ambiguous_features_in_query': {}}

In [59]:
feature_view._query._left_features

[Feature('idade', 'bigint', None, False, False, False, None, None, 52432),
 Feature('dias_desde_ultimo_login', 'bigint', None, False, False, False, None, None, 52432)]

In [39]:
#feature_view.get_inference_data()

In [64]:
df = feature_view.get_batch_data()

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (3.38s) 


In [65]:
df

,pessoa_id,idade,dias_desde_ultimo_login,gastos_100d,peso,quantos_dias_sem_comer
0,1001,28,1,1200.0,110,1
1,1002,45,5,8500.0,68,2
2,1003,32,2,400.0,90,4
3,1004,19,0,NaN,62,3


In [42]:
X_train, X_test, y_train, y_test = feature_view.train_test_split(
    test_size=0.2,
    description="Treino do modelo de risco v1"
)

2026-09-03 15:05:07,160 INFO: Computing insert statistics
2026-09-03 15:05:07,173 INFO: Computing insert statistics


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (4.19s) 


In [46]:
print("--- [OFFLINE] Dados de Treino (X_train) ---")
print(X_train)

--- [OFFLINE] Dados de Treino (X_train) ---
   idade  dias_desde_ultimo_login  gastos_100d  peso  quantos_dias_sem_comer
1     45                        5       8500.0    68                       2
2     32                        2        400.0    90                       4
3     19                        0          NaN    62                       3


In [40]:
#feature_view.init_online_vector_service()

In [74]:
df
feature_vectors = feature_view.get_feature_vectors(
    entry=[{"pessoa_id": 1001, "pessoa_id":1003}, {"pessoa_id": 1002}, {"pessoa_id":1001}],
    passed_features=[{"idade":30}, {"idade":33}, {"idade":46}]
)
feature_vectors

,pessoa_id,idade,dias_desde_ultimo_login,gastos_100d,peso,quantos_dias_sem_comer
0,1001,28,1,1200.0,110,1
1,1002,45,5,8500.0,68,2
2,1003,32,2,400.0,90,4
3,1004,19,0,NaN,62,3


[[1003, 30, 2, 400.0, 90, 4],
 [1002, 33, 5, 8500.0, 68, 2],
 [1001, 46, 1, 1200.0, 110, 1]]

In [59]:
feature_vectors

[[32, 2, 400.0, 90, 4], [45, 5, 8500.0, 68, 2]]

In [75]:
X_train, y_train, X_test, y_test = feature_view.get_train_test_split(training_dataset_version=1)

RestAPIError: Metadata operation error: (url: https://eu-west.cloud.hopsworks.ai/hopsworks-api/api/project/42213/featurestores/30898/featureview/modelo_forecasting/version/2/trainingdatasets/version/1). Server response: 
HTTP code: 404, HTTP reason: Not Found, body: b'{"errorCode":270012,"usrMsg":"FeatureView name: modelo_forecasting, version: 2, td version: 1","errorMsg":"Training dataset wasn\'t found."}', error code: 270012, error msg: Training dataset wasn't found., user msg: FeatureView name: modelo_forecasting, version: 2, td version: 1